# ScholarGuard — Stage 4: Train the Real-vs-AI Artifact Classifier

Fine-tunes a **MobileNetV3-small** binary classifier (`real` vs `ai_generated`) on a **free Colab/Kaggle T4 GPU**, then exports weights to drop into `src/models/weights/artifact_classifier.pt` for local inference.

**Why this runs here and not in the main codebase:** training needs a GPU; the ScholarGuard repo is CPU-only by design. The architecture in this notebook is identical to `src/models/artifact_classifier.py::build_model` — keep the two in sync.

**Expected training time on a free T4:** ~3–6 minutes for 8 epochs on a few thousand 224×224 images.

### Steps
1. Install deps & check the GPU
2. Provide data (Google Drive mount **or** upload)
3. Build datasets / dataloaders with light augmentation
4. Build the model (ImageNet-pretrained MobileNetV3-small, new 2-class head)
5. Fine-tune + evaluate on a held-out split
6. Export the checkpoint and download it

## 1. Install dependencies and confirm the GPU

In [ ]:
!pip -q install torch torchvision scikit-learn
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU (T4)'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Provide the data

Expected layout (two flat folders of images):
```
DATA_ROOT/
  real_captured_samples/   # genuine microscopy / blot / gel scans
  ai_generated_samples/    # GAN/diffusion-generated scientific-looking images
```

**Option A — Google Drive (recommended):** upload the two folders to Drive, mount it, set `DATA_ROOT`.

**Option B — direct upload:** zip the two folders, upload, unzip (see the commented cell).

> For a first end-to-end dry run you can generate synthetic stand-ins with the repo's `src/utils/synth.py --ai` (as used by the local pipeline). **For a classifier that generalizes to real fraud, use genuine captured images and real generator output**, not the synthetic stand-ins.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/scholarguard/data'  # <-- edit me

# --- Option B: direct upload instead of Drive ---
# from google.colab import files; import zipfile, os
# up = files.upload()                       # choose data.zip
# with zipfile.ZipFile(next(iter(up))) as z: z.extractall('/content/data')
# DATA_ROOT = '/content/data'

import os
REAL_DIR = os.path.join(DATA_ROOT, 'real_captured_samples')
AI_DIR   = os.path.join(DATA_ROOT, 'ai_generated_samples')
print('real:', len(os.listdir(REAL_DIR)), '| ai:', len(os.listdir(AI_DIR)))

## 3. Datasets and dataloaders

Preprocessing **must match** `src/models/artifact_classifier.py`: 224×224, ImageNet normalization. Training adds light flips/rotation/jitter; validation does not. An 85/15 stratified split gives a held-out set.

In [ ]:
import glob
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

CLASSES = ('real', 'ai_generated')            # index 0 = real, 1 = ai_generated
INPUT_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
EXTS = ('*.png', '*.jpg', '*.jpeg', '*.tif', '*.tiff', '*.bmp')

def list_dir(folder):
    files = []
    for ext in EXTS:
        files += glob.glob(os.path.join(folder, ext))
    return sorted(files)

paths  = list_dir(REAL_DIR) + list_dir(AI_DIR)
labels = [0]*len(list_dir(REAL_DIR)) + [1]*len(list_dir(AI_DIR))
tr_p, va_p, tr_y, va_y = train_test_split(paths, labels, test_size=0.15,
                                          stratify=labels, random_state=42)
print(f'train={len(tr_p)}  val={len(va_p)}')

train_tf = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.1, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class FigureDataset(Dataset):
    def __init__(self, paths, labels, tf):
        self.paths, self.labels, self.tf = paths, labels, tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        return self.tf(img), self.labels[i]

train_dl = DataLoader(FigureDataset(tr_p, tr_y, train_tf), batch_size=32,
                      shuffle=True, num_workers=2)
val_dl   = DataLoader(FigureDataset(va_p, va_y, val_tf), batch_size=64,
                      shuffle=False, num_workers=2)

## 4. Build the model

**This architecture is identical to `src/models/artifact_classifier.py::build_model`.** If you change the backbone here, change it there too, and record it in the checkpoint's `backbone` field.

In [ ]:
import torch.nn as nn
from torchvision import models

BACKBONE = 'mobilenet_v3_small'

def build_model(backbone=BACKBONE, num_classes=2):
    net = models.mobilenet_v3_small(
        weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    in_features = net.classifier[3].in_features
    net.classifier[3] = nn.Linear(in_features, num_classes)
    return net

device = 'cuda'
model = build_model().to(device)
print(sum(p.numel() for p in model.parameters())/1e6, 'M params')

## 5. Fine-tune and evaluate

AdamW + cosine schedule, cross-entropy. 8 epochs is plenty for a 2-class transfer-learning task; watch the held-out accuracy and stop early if it plateaus.

In [ ]:
EPOCHS = 8
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
crit = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate():
    model.eval(); correct = total = 0
    for x, y in val_dl:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item(); total += y.numel()
    return correct / total

best_acc = 0.0
for epoch in range(EPOCHS):
    model.train(); running = 0.0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = crit(model(x), y)
        loss.backward(); opt.step()
        running += loss.item() * x.size(0)
    sched.step()
    acc = evaluate(); best_acc = max(best_acc, acc)
    print(f'epoch {epoch+1}/{EPOCHS}  loss={running/len(tr_p):.4f}  val_acc={acc:.4f}')
print('best val accuracy:', best_acc)

In [ ]:
# Detailed held-out metrics (precision/recall/confusion)
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
model.eval(); preds, gts = [], []
with torch.no_grad():
    for x, y in val_dl:
        preds += model(x.to(device)).argmax(1).cpu().tolist(); gts += y.tolist()
print(confusion_matrix(gts, preds))
print(classification_report(gts, preds, target_names=CLASSES))

## 6. Export the checkpoint

The checkpoint schema matches what `classify_artifact` expects: `state_dict`, `backbone`, `input_size`, `classes`, `val_accuracy`. Download `artifact_classifier.pt` and place it in the repo at **`src/models/weights/artifact_classifier.pt`** — the local detector will pick it up automatically.

In [ ]:
ckpt = {
    'state_dict': model.state_dict(),
    'backbone': BACKBONE,
    'input_size': INPUT_SIZE,
    'classes': list(CLASSES),
    'val_accuracy': float(best_acc),
    'normalization': {'mean': MEAN, 'std': STD},
}
torch.save(ckpt, 'artifact_classifier.pt')
print('saved artifact_classifier.pt  (val_acc=%.4f)' % best_acc)

from google.colab import files
files.download('artifact_classifier.pt')
# Then in the repo:  mkdir -p src/models/weights
#                    mv ~/Downloads/artifact_classifier.pt src/models/weights/